# 03 — NDVI-Guided Sampling

Create sample points from NDVI classes and keep only points that fall inside the EMIT observation footprint.

Default thresholds reproduce the strict sampling used in the existing notebook:

- Stress: 0.25–0.30
- Moderate: 0.40–0.45
- Healthy: ≥0.52

Change the thresholds in the configuration cell when your study design uses different limits.

In [ ]:
from pathlib import Path
import random
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import Point

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NDVI_FILE = DATA_DIR / "NDVI_2023.tif"
EMIT_FILE = DATA_DIR / "EMIT_ortho.dat"
OUT_FILE = OUTPUT_DIR / "NDVI_Selected_Points.shp"

STRESS_MIN, STRESS_MAX = 0.25, 0.30
MODERATE_MIN, MODERATE_MAX = 0.40, 0.45
HEALTHY_MIN = 0.52

N_PER_CLASS = 3
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

In [ ]:
if not NDVI_FILE.exists():
    raise FileNotFoundError(f"Missing NDVI raster: {NDVI_FILE.resolve()}")
if not EMIT_FILE.exists():
    raise FileNotFoundError(f"Missing EMIT raster: {EMIT_FILE.resolve()}")

with rasterio.open(NDVI_FILE) as ndvi_src, rasterio.open(EMIT_FILE) as emit_src:
    ndvi = ndvi_src.read(1)
    emit_bounds = emit_src.bounds

    print("NDVI CRS:", ndvi_src.crs)
    print("EMIT CRS:", emit_src.crs)

    # Work in the EMIT CRS for point/footprint testing.
    if ndvi_src.crs != emit_src.crs:
        ndvi_gdf = gpd.GeoDataFrame(
            geometry=[Point(0, 0)], crs=ndvi_src.crs
        ).to_crs(emit_src.crs)
        del ndvi_gdf

    from pyproj import Transformer
    transformer = Transformer.from_crs(
        ndvi_src.crs, emit_src.crs, always_xy=True
    )

    stress_pts = []
    moderate_pts = []
    healthy_pts = []

    rows, cols = ndvi.shape

    # Step=5 follows the existing notebook's speed-up approach.
    for r in range(0, rows, 5):
        for c in range(0, cols, 5):
            value = ndvi[r, c]

            if not np.isfinite(value):
                continue

            x0, y0 = ndvi_src.xy(r, c)
            x, y = transformer.transform(x0, y0)

            if not (
                emit_bounds.left <= x <= emit_bounds.right
                and emit_bounds.bottom <= y <= emit_bounds.top
            ):
                continue

            if STRESS_MIN <= value <= STRESS_MAX:
                stress_pts.append((x, y, value))
            elif MODERATE_MIN <= value <= MODERATE_MAX:
                moderate_pts.append((x, y, value))
            elif value >= HEALTHY_MIN:
                healthy_pts.append((x, y, value))

print("Candidates:")
print("Stress:", len(stress_pts))
print("Moderate:", len(moderate_pts))
print("Healthy:", len(healthy_pts))

In [ ]:
if len(stress_pts) < N_PER_CLASS or len(moderate_pts) < N_PER_CLASS or len(healthy_pts) < N_PER_CLASS:
    raise ValueError("Not enough candidate points for the requested sample size.")

stress_sel = random.sample(stress_pts, N_PER_CLASS)
moderate_sel = random.sample(moderate_pts, N_PER_CLASS)

# Existing workflow selected the highest NDVI healthy points.
healthy_pts = sorted(healthy_pts, key=lambda item: item[2], reverse=True)
healthy_sel = healthy_pts[:N_PER_CLASS]

records = (
    [("Stress", *p) for p in stress_sel]
    + [("Moderate", *p) for p in moderate_sel]
    + [("Healthy", *p) for p in healthy_sel]
)

gdf = gpd.GeoDataFrame(
    records,
    columns=["Class", "X", "Y", "NDVI"],
    geometry=[Point(x, y) for _, x, y, _ in records],
    crs=emit_src.crs,
)

gdf.to_file(OUT_FILE)
print("Saved:", OUT_FILE.resolve())
print(gdf[["Class", "NDVI"]])